In [1]:
import json
import numpy as np
import pandas as pd
from scipy.optimize import linprog

In [ ]:
# 1. CARGAR PARÁMETROS DEL HMM Y COSTOS DE DECISIÓN
# =================================================
with open("../datos/resultados_hmm.json", "r", encoding="utf-8") as f:
    datos_hmm = json.load(f)

A = np.array(datos_hmm["matriz_transicion"])

# Matriz de Costos Operativos C(Acción Defensor, Estado Red) usada en Teoría de Decisiones
# Filas: [d1: Monitoreo Pasivo, d2: Inspección Profunda, d3: Bloqueo Total]
# Columnas: [E0: Reconnaissance, E1: Initial Access, E2: Exfiltration, E3: Pivote]
C = np.array([
    [0,   15, 100, 30],  # d1: Monitoreo Pasivo
    [5,    8,  45, 10],  # d2: Inspección Profunda
    [20,  12,   5,  8]   # d3: Bloqueo Total
])

In [8]:
# 2. FORMULACIÓN DEL JUEGO DE SUMA CERO (DEFENSOR VS. ATACANTE)
# =============================================================
# El Defensor elige su contramedida (filas): d1, d2, d3.
# El Atacante elige su perfil de operación (columnas):
#   - a1 (Sigiloso / Reconocimiento): opera siguiendo la dinámica de E0
#   - a2 (Infiltración / Pivote): opera siguiendo una mezcla de E1 y E3 (50% y 50%)
#   - a3 (Exfiltración Agresiva): fuerza la dinámica de E2

estrategias_defensor = ["D1: Monitoreo Pasivo", "D2: Inspección Profunda", "D3: Bloqueo Total"]
estrategias_atacante = ["A1: Sigiloso (E0)", "A2: Infiltración/Pivote (E1+E3)", "A3: Exfiltración Agresiva (E2)"]

# Distribuciones de transición asociadas a cada estrategia del Atacante
p_A1 = A[0]                      # Fila E0
p_A2 = 0.5 * A[1] + 0.5 * A[3]   # Promedio filas E1 y E3
p_A3 = A[2]                      # Fila E2

# Matriz de Pagos M (Pérdida esperada del Defensor = Ganancia esperada del Atacante)
M = np.column_stack([
    C @ p_A1,
    C @ p_A2,
    C @ p_A3
])

df_juego = pd.DataFrame(M, index=estrategias_defensor, columns=estrategias_atacante)
print("--- Matriz de Pagos del Juego (Pérdida del Defensor / Ganancia del Atacante) ---")
print(df_juego.round(2))

--- Matriz de Pagos del Juego (Pérdida del Defensor / Ganancia del Atacante) ---
                         A1: Sigiloso (E0)  A2: Infiltración/Pivote (E1+E3)  \
D1: Monitoreo Pasivo                  1.20                            19.64   
D2: Inspección Profunda               5.33                             8.58   
D3: Bloqueo Total                    19.65                            11.20   

                         A3: Exfiltración Agresiva (E2)  
D1: Monitoreo Pasivo                              79.01  
D2: Inspección Profunda                           36.50  
D3: Bloqueo Total                                  8.02  


In [5]:
# ==============================================================================
# 3. VERIFICACIÓN DE EQUILIBRIO DE NASH EN ESTRATEGIAS PURAS (Criterio Minimax)
# ==============================================================================
max_por_fila = M.max(axis=1)  # Peor escenario para el Defensor en cada acción
min_por_col = M.min(axis=0)   # Peor escenario para el Atacante en cada estrategia

valor_minimax = max_por_fila.min()  # Cota superior del juego (Defensor minimiza su pérdida máxima)
valor_maximin = min_por_col.max()   # Cota inferior del juego (Atacante maximiza su ganancia mínima)

print(f"\nValor Minimax (Defensor): {valor_minimax:.4f}")
print(f"Valor Maximin (Atacante): {valor_maximin:.4f}")

if np.isclose(valor_minimax, valor_maximin):
    print("-> Existe un Equilibrio de Nash en Estrategias Puras (Punto de Silla).")
else:
    print("-> NO existe Punto de Silla. Se requiere Equilibrio de Nash en Estrategias Mixtas.")


Valor Minimax (Defensor): 19.6490
Valor Maximin (Atacante): 8.5771
-> NO existe Punto de Silla. Se requiere Equilibrio de Nash en Estrategias Mixtas.


In [6]:
# ==============================================================================
# 4. CÁLCULO DEL EQUILIBRIO DE NASH EN ESTRATEGIAS MIXTAS (Programación Lineal)
# ==============================================================================
# Problema del Defensor: Minimizar v (pérdida máxima esperada)
# Sujeto a: M^T * p <= v   =>   M^T * p - v <= 0
#           sum(p) = 1, p >= 0
# Variables de decisión: [p1, p2, p3, v]

n_def, n_ata = M.shape
c_def = np.zeros(n_def + 1)
c_def[-1] = 1.0  # Queremos minimizar v

# Restricciones de desigualdad: M^T * p - v <= 0
A_ub_def = np.hstack([M.T, -np.ones((n_ata, 1))])
b_ub_def = np.zeros(n_ata)

# Restricción de igualdad: p1 + p2 + p3 = 1
A_eq_def = np.zeros((1, n_def + 1))
A_eq_def[0, :n_def] = 1.0
b_eq_def = np.array([1.0])

bounds_def = [(0, 1)] * n_def + [(None, None)]
res_def = linprog(c_def, A_ub=A_ub_def, b_ub=b_ub_def, A_eq=A_eq_def, b_eq=b_eq_def, bounds=bounds_def, method="highs")

prob_defensor = res_def.x[:n_def]
valor_juego = res_def.x[-1]

# Problema del Atacante (Dual): Maximizar w (ganancia mínima esperada) <=> Minimizar -w
# Sujeto a: M * q >= w   =>   -M * q + w <= 0
#           sum(q) = 1, q >= 0
c_ata = np.zeros(n_ata + 1)
c_ata[-1] = -1.0  # Maximizar w

A_ub_ata = np.hstack([-M, np.ones((n_def, 1))])
b_ub_ata = np.zeros(n_def)

A_eq_ata = np.zeros((1, n_ata + 1))
A_eq_ata[0, :n_ata] = 1.0
b_eq_ata = np.array([1.0])

bounds_ata = [(0, 1)] * n_ata + [(None, None)]
res_ata = linprog(c_ata, A_ub=A_ub_ata, b_ub=b_ub_ata, A_eq=A_eq_ata, b_eq=b_eq_ata, bounds=bounds_ata, method="highs")

prob_atacante = res_ata.x[:n_ata]

In [7]:
# ==============================================================================
# 5. RESULTADOS FINALES DEL EQUILIBRIO DE NASH
# ==============================================================================
print("\n========================================================")
print(f"VALOR DEL JUEGO (Costo Esperado en Equilibrio): {valor_juego:.4f}")
print("========================================================")

print("\n--- Estrategia Mixta Óptima del DEFENSOR ---")
for est, prob in zip(estrategias_defensor, prob_defensor):
    print(f"  {est:28s}: {prob*100:6.2f}%")

print("\n--- Estrategia Mixta Óptima del ATACANTE ---")
for est, prob in zip(estrategias_atacante, prob_atacante):
    print(f"  {est:35s}: {prob*100:6.2f}%")


VALOR DEL JUEGO (Costo Esperado en Equilibrio): 15.7573

--- Estrategia Mixta Óptima del DEFENSOR ---
  D1: Monitoreo Pasivo        :   0.00%
  D2: Inspección Profunda     :  27.17%
  D3: Bloqueo Total           :  72.83%

--- Estrategia Mixta Óptima del ATACANTE ---
  A1: Sigiloso (E0)                  :  66.54%
  A2: Infiltración/Pivote (E1+E3)    :   0.00%
  A3: Exfiltración Agresiva (E2)     :  33.46%
